# P2P / DyNECT — standalone 4-hour reduction audit

**No earlier notebook or shared Jupyter kernel is required.** Run top to bottom from the project root or the `p2p_dynect` subfolder. The first cells locate your P2P and DyNECT packages, build one **24-hour, 30-minute** input day (`2016-01-07`), aggregate it to **six 4-hour periods**, and define the feeder/local-constraint reduction helpers.

The reference 4-hour vGNE, reduced vGNE, and independent centralized optimization are computed **at identical resolution**. No full-resolution 48-hour solve and no `GNESetSolver` enumeration are launched.

**Scientific caveats:** 4-hour aggregation changes EV events and the time discretization; the legacy centralized feeder includes reactive power whereas the DyNECT adapter is active-power-only. Feeder screening is certified only under the injection bounds implemented below, which you should review for your case. Costs/dispatches are empirical comparisons, not guaranteed coincidence.


## 0. Load one day and define the necessary helpers

In [1]:
import sys, copy, math, time, itertools
from pathlib import Path
from dataclasses import replace

import numpy as np
import pandas as pd
from scipy.optimize import linprog
from IPython.display import display

DATE = "2016-01-07"
SEED = 42
SOURCE_DT = 0.5
TARGET_DT = 4.0

# Look upward for the original project, even if notebook lives in p2p_dynect/.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "scenarios.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Cannot find scenarios.py. Open notebook in your P2P repo or its p2p_dynect subfolder.")

# Favor an existing p2p package in the notebook's directory, then original imports.
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(HERE))
DYNECT_REPO = Path.home() / "Documents/Github/dynect_py_p2p"
if DYNECT_REPO.is_dir():
    sys.path.insert(0, str(DYNECT_REPO))

import config
from scenarios import make_scenario_horizon
from dynect import DynLQGameTV
from p2p.setup import (
    build_game, load_ieee33, _feeder_maps, unpack_controls,
    evaluated_costs, feeder_results, rollout_states,
)
from p2p.vgne import solve_vgne

print("Original P2P repo:", ROOT)
print("DyNECT source:", __import__("dynect").__file__)
print("P2P adapter source:", __import__("p2p.setup", fromlist=["x"]).__file__)
print("Experiment:", DATE, "population:", config.PROSUMER_SCENARIO)

# Build the source day once, without the 48-hour lookahead.
# (This does not run any optimization.)
short_data = make_scenario_horizon(
    DATE, seed=SEED, resolution_h=SOURCE_DT, lookahead_h=0
)
grid = load_ieee33()
assert len(short_data["buy_price"]) == 48, "Expected 24h of half-hour source data"
print("Source:", len(short_data["buy_price"]), "steps ×", short_data["dt"], "h")


Original P2P repo: /Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative
DyNECT source: /Users/catharinahente/Documents/Github/dynect_py_p2p/dynect/__init__.py
P2P adapter source: /Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative/p2p_dynect/p2p/setup.py
Experiment: 2016-01-07 population: 4
Building winter scenario (T=48, dt=0.5h, resolution=30min)...
  Prices 2016: loaded from cache
  P1: Residential ++ (bus 5)
    HP: Air_Alternative_1 (Air alternative, Hanover)
  P2: Office building (bus 11)
    ⚠ No EV activity detected — using fallback schedule
  P3: Residential HP only (bus 17)
    HP: Air_Semi-Parallel_1 (Air semi-parallel, Hanover)
  P4: Residential PV only (bus 32)
  Done. N=4, T=48, dt=0.5h, max_demand=16.3 kW
Source: 48 steps × 0.5 h


In [2]:
def clean_local(A, b):
    A = np.asarray(A, float)
    b = np.asarray(b, float)
    best = {}
    for row, rhs in zip(A, b):
        row = row.copy()
        row[row == 0.] = 0.  # normalize -0.0, no numerical rounding
        if not np.any(row):
            if rhs < 0:
                raise ValueError('Infeasible identically-zero local row')
            continue
        key = row.tobytes()   # only EXACT identical normals
        if key not in best or rhs < best[key][1]:
            best[key] = (row, rhs)
    if not best:
        return np.zeros((0, A.shape[1])), np.zeros(0)
    return np.vstack([v[0] for v in best.values()]), np.array([v[1] for v in best.values()])

GAME_FIELDS = ['A','B','Q','R','P','c','q','r','p',
               'C_x','b_x','C_loc','b_loc','C_u','b_u']

def rebuild_constraints(base, *, keep_labels=None, clean_locals=False,
                        localize_single_agent=False):
    """Create a new DynLQGameTV, preserving dynamics/costs and control ordering.

    Shared rows can be removed by their labels (after an independent proof of
    redundancy), or moved to local constraints when they involve one agent.
    """
    old = base.game
    kw = {k: getattr(old,k) for k in GAME_FIELDS}
    labels = list(base.shared_labels)
    if keep_labels is None:
        keep_labels = labels
    if len(set(keep_labels)) != len(keep_labels):
        raise ValueError('Shared label names must be unique')
    indices = [j for j,label in enumerate(labels) if label in set(keep_labels)]
    if len(indices) != len(keep_labels):
        raise ValueError('Unknown requested shared label')
    c_loc, b_loc, c_u, b_u = [], [], [], []
    kept_labels = [labels[j] for j in indices]
    moved = []
    ownership = {}
    if localize_single_agent:
        for j in indices:
            owners = [i for i in range(base.N) if any(
                np.any(old.C_u[t][i][j] != 0) for t in range(base.T))]
            ownership[j] = owners
            if len(owners) == 1:
                moved.append(j)
            elif len(owners) == 0:
                if any(old.b_u[t][j] < 0 for t in range(base.T)):
                    raise ValueError(f'Constant infeasible shared row: {labels[j]}')
                moved.append(j)
        indices = [j for j in indices if j not in moved]
        kept_labels = [labels[j] for j in indices]
    for t in range(base.T):
        local_a = [np.asarray(old.C_loc[t][i]).copy() for i in range(base.N)]
        local_b = [np.asarray(old.b_loc[t][i]).copy() for i in range(base.N)]
        if localize_single_agent:
            for j in moved:
                owners = ownership[j]
                if len(owners) == 1:
                    i = owners[0]
                    local_a[i] = np.vstack((local_a[i], old.C_u[t][i][j:j+1]))
                    local_b[i] = np.append(local_b[i], old.b_u[t][j])
        if clean_locals:
            cleaned = [clean_local(a,b) for a,b in zip(local_a,local_b)]
            local_a = [x[0] for x in cleaned]
            local_b = [x[1] for x in cleaned]
        c_loc.append(local_a); b_loc.append(local_b)
        c_u.append([old.C_u[t][i][indices,:].copy() for i in range(base.N)])
        b_u.append(old.b_u[t][indices].copy())
    kw.update(C_loc=c_loc, b_loc=b_loc, C_u=c_u, b_u=b_u)
    new = replace(base, game=DynLQGameTV(**kw), shared_labels=kept_labels)
    return new, [labels[j] for j in moved]

def model_max_violation(case, u):
    """Check a candidate against the given (possibly FULL) dynamic game's rows,
    without constructing an additional enormous condensed AVI.
    """
    controls = unpack_controls(case, u)
    states = rollout_states(case, u)
    worst = -np.inf
    for t in range(case.T):
        shared = np.zeros(len(case.game.b_u[t]))
        for i in range(case.N):
            ui = np.array([controls[i][name][t] for name in case.controls[i]])
            shared += case.game.C_u[t][i] @ ui
            A = case.game.C_loc[t][i]
            if A.shape[0]:
                worst = max(worst,float(np.max(A @ ui - case.game.b_loc[t][i])))
        if shared.size:
            worst = max(worst,float(np.max(shared-case.game.b_u[t])))
        if case.game.C_x[t].shape[0]:
            worst = max(worst,float(np.max(case.game.C_x[t]@states[t+1]-case.game.b_x[t])))
    return max(0.,float(worst))

In [3]:
def injection_intervals(data):
    N, T = len(data['demand']), len(data['buy_price'])
    lo, hi = np.zeros((N,T)), np.zeros((N,T))
    base = np.asarray(data['generation'],float) - np.asarray(data['demand'],float)
    available = np.asarray(data['ev_availability'],float)
    for i in range(N):
        batt_limit = min(float(data['battery_power'][i]),
                         float(data['battery_smax'][i]) *
                         math.cos(math.pi/float(data.get('battery_polygon_sides',8))))
        ev_limit = min(float(data['ev_power'][i]),
                       float(data['ev_smax'][i]) *
                       math.cos(math.pi/float(data.get('ev_polygon_sides',8))))
        ev_ch = ev_limit*available[i]
        ev_dis = ev_ch * float(bool(data.get('ev_v2g',True)))
        lo[i] = base[i] - batt_limit - ev_ch - float(data['hp_max_power'][i])
        hi[i] = base[i] + batt_limit + ev_dis
    return lo, hi

def certify_feeder_constraints(case, margin=1e-8):
    """Safe global screening: returns per-shared-label interval upper bound on
    maximum constraint violation. Only negative *certified* bounds are removed.
    MW and squared-pu rows are compared in their native units.
    """
    if case.grid is None:
        raise ValueError('Need original feeder to certify feasibility')
    g, d = case.grid, case.data
    lo,hi = injection_intervals(d)
    f0,L,drop = _feeder_maps(g,list(map(int,d['agent_bus'])))
    a = L[:,:,None] * lo[None,:,:]
    b = L[:,:,None] * hi[None,:,:]
    flow_min = f0[:,None] + np.minimum(a,b).sum(axis=1)
    flow_max = f0[:,None] + np.maximum(a,b).sum(axis=1)
    vm = -drop @ L
    v0 = float(g['root_voltage'])**2 - drop @ f0
    va = vm[:,:,None]*lo[None,:,:]
    vb = vm[:,:,None]*hi[None,:,:]
    v_min = v0[:,None] + np.minimum(va,vb).sum(axis=1)
    v_max = v0[:,None] + np.maximum(va,vb).sum(axis=1)
    lim = float(d['line_limit_mva'])*math.cos(math.pi/8)
    bound = {}
    for k,e in enumerate(g['branches']):
        bound[f'flow_{e}_upper'] = float(np.max(flow_max[k] - lim))
        bound[f'flow_{e}_lower'] = float(np.max(-lim-flow_min[k]))
    for bus in range(g['bus_count']):
        if bus == g['root']:
            continue
        bound[f'voltage_{bus}_upper'] = float(np.max(v_max[bus]-g['voltage_max'][bus]**2))
        bound[f'voltage_{bus}_lower'] = float(np.max(g['voltage_min'][bus]**2-v_min[bus]))
    records = []
    for label in case.shared_labels:
        if label.startswith('edge_'):
            records.append({'Label':label,'Type':'P2P',
                            'Worst possible violation (bound)':np.nan,
                            'Certified redundant':False})
        else:
            if label not in bound:
                raise KeyError(f'Unknown feeder label: {label}')
            records.append({'Label':label,'Type':'flow' if label.startswith('flow_') else 'voltage',
                            'Worst possible violation (bound)':bound[label],
                            'Certified redundant':bound[label] < -margin})
    return pd.DataFrame(records)

In [4]:
def aggregate_time_grid(data, target_dt=2.0):
    d = copy.deepcopy(data)
    old_dt = float(d['dt'])
    factor = int(round(target_dt/old_dt))
    if factor < 1 or not np.isclose(factor*old_dt,target_dt):
        raise ValueError('New dt must be an integer multiple of original dt')
    oldT = len(d['buy_price'])
    if oldT % factor or int(d['T_eval']) % factor:
        raise ValueError('Horizon and evaluation length must be divisible by aggregation')
    N = len(d['demand']); T=oldT//factor
    def agent_agg(key,op):
        a = np.asarray(d[key],float).reshape(N,T,factor)
        d[key]=a.mean(axis=2) if op=='mean' else a.sum(axis=2)
    for k in ('demand','generation','hp_base_power','hp_q_heat'):
        agent_agg(k,'mean')
    agent_agg('ev_driving_load','sum')   # EV trip consumption is ENERGY per bin
    availability=np.asarray(d['ev_availability'],int).reshape(N,T,factor)
    mixed=((availability.min(axis=2) != availability.max(axis=2)))
    d['ev_availability']=availability.min(axis=2).astype(int)
    for k in ('buy_price','sell_price','p2p_price'):
        d[k]=np.asarray(d[k],float).reshape(T,factor).mean(axis=1)
    d['dt']=float(target_dt);d['resolution_h']=float(target_dt)
    d['T']=T;d['T_eval']=int(data['T_eval'])//factor
    return d, mixed

def solve_compact(case, verbose=False):
    result = solve_vgne(case, verbose=verbose)
    result.pop("mp", None)
    result.pop("avi", None)
    return result


## 1. Aggregate to 6 × 4 hours, then audit devices and constraints

In [5]:
data4,mixed4 = aggregate_time_grid(short_data, target_dt=4.0)
assert len(data4['buy_price']) == 6 and np.isclose(data4['dt'],4.)
assert int(data4['T_eval']) == 6
edges4 = [tuple(sorted(e)) for e in data4['edges']]
print('4h steps:',len(data4['buy_price']))
print('Mixed EV availability bins:', int(np.sum(mixed4)))
if np.any(mixed4): print('WARNING: coarse EV scheduling may change physical outcomes/departure timing.')

4h steps: 6
Mixed EV availability bins: 3


## 1 — Are controls/constraints created for absent devices?

`p2p.setup._layout_for_agent` always allocates seven base controls per prosumer, regardless of device ownership. BESS/EV/HP **states**, however, are instantiated only if the corresponding capacity is positive. The absent controls are constrained to zero by per-period local bounds, so their rows are present until eliminated. EV availability can also force controls to zero for individual periods; these are **not** all-horizon zeros.

In [6]:
D = data4
device4 = build_game(D, grid=None)
rows = []
for i in range(len(D['demand'])):
    rows.append(dict(prosumer=f'P{i+1}',
        BESS_kWh=float(D['battery_capacity'][i]),
        EV_kWh=float(D['ev_capacity'][i]),
        HP_kW=float(D['hp_max_power'][i]),
        BESS_state=((i,'bess') in device4.state_index),
        EV_state=((i,'ev') in device4.state_index),
        HP_state=((i,'hp') in device4.state_index),
        EV_available_4h_steps=int(np.sum(D['ev_availability'][i]))))
display(pd.DataFrame(rows))

,prosumer,BESS_kWh,EV_kWh,HP_kW,BESS_state,EV_state,HP_state,EV_available_4h_steps
0,P1,15.0,60.0,10.0,True,True,True,0
1,P2,20.0,40.0,0.0,True,True,False,2
2,P3,0.0,0.0,8.0,False,False,True,6
3,P4,10.0,0.0,0.0,True,False,False,6


## 2 — Same-resolution vGNE: full feeder versus certified-screened feeder

This is the exact-reduction benchmark. Coarsening from 2h to 4h is **a separate approximation**; never call it an exact reduction. Network screening must be re-certified at 4h, not inherited blindly from the 2h case.

In [7]:
full4 = build_game(copy.deepcopy(data4),grid=grid,clearing_epsilon_kw=0.)
report4 = certify_feeder_constraints(full4)
keep4 = report4.loc[~report4['Certified redundant'],'Label'].tolist()
reduced4, moved4 = rebuild_constraints(full4,keep_labels=keep4,
                                       clean_locals=True,localize_single_agent=True)
print('Feeder screening:')
display(report4.groupby(['Type','Certified redundant']).size().rename('rows per step').to_frame())
print('Shared rows/step:', len(full4.shared_labels),'→',len(reduced4.shared_labels))
print('β dimension:', (full4.N-1)*full4.T*full4.game.m_u,
      '→',(reduced4.N-1)*reduced4.T*reduced4.game.m_u)

start=time.perf_counter(); sol_full4=solve_compact(full4,verbose=False)
print('Full vGNE time:',round(time.perf_counter()-start,2),'s')
start=time.perf_counter(); sol_reduced4=solve_compact(reduced4,verbose=False)
print('Reduced vGNE time:',round(time.perf_counter()-start,2),'s')
J_full4=evaluated_costs(full4,sol_full4['u'])
J_reduced4=evaluated_costs(reduced4,sol_reduced4['u'])
display(pd.DataFrame({'Prosumer':[f'P{i+1}' for i in range(full4.N)],
   'Full feeder vGNE [EUR]':J_full4,
   'Screened feeder vGNE [EUR]':J_reduced4,
   'Difference [EUR]':J_reduced4-J_full4}).round(7))
print('Max cost Δ:',np.max(abs(J_reduced4-J_full4)))
print('Control RMS Δ:',np.sqrt(np.mean((sol_full4['u']-sol_reduced4['u'])**2)))
print('Screened dispatch violation in FULL feeder game:',
      model_max_violation(full4,sol_reduced4['u']))

Feeder screening:


,,rows per step
Type,Certified redundant,
P2P,False,8
flow,True,64
voltage,True,64


Shared rows/step: 136 → 8
β dimension: 2448 → 144
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2806500
Academic license 2806500 - for non-commercial use only - registered to ch___@ethz.ch
Full vGNE time: 0.65 s
Reduced vGNE time: 0.02 s


,Prosumer,Full feeder vGNE [EUR],Screened feeder vGNE [EUR],Difference [EUR]
0,P1,11.050777,11.050777,0.0
1,P2,54.985863,54.985863,0.0
2,P3,3.321077,3.321077,0.0
3,P4,10.859313,10.859313,0.0


Max cost Δ: 0.0
Control RMS Δ: 0.0
Screened dispatch violation in FULL feeder game: 3.5416114485542494e-14


## 3 — Independent centralized Gurobi solve at 4h

Legacy `algo_central.centralized` includes reactive-power aspects absent from the active-only DyNECT adapter; coincidence is *empirical*, not guaranteed. Its objective and valuation conventions can differ. Never compare the new four-hour case to saved 30-minute central costs as a like-for-like check.

In [8]:
from algo_central import centralized
try:
    start=time.perf_counter()
    central4=centralized(copy.deepcopy(data4))
    J_central4=np.asarray(central4['costs'],float)
    print('Legacy centralized 4h time:',round(time.perf_counter()-start,2),'s')
    display(pd.DataFrame({'Prosumer':[f'P{i+1}' for i in range(full4.N)],
      'Centralized 4h [EUR]':J_central4,
      'Full feeder vGNE 4h [EUR]':J_full4,
      'Screened vGNE 4h [EUR]':J_reduced4,
      'Screened − centralized [EUR]':J_reduced4-J_central4}).round(6))
    physical=unpack_controls(full4,sol_full4['u'])
    comparison=[]
    for name in ('buy','sell','charge','discharge','ev_charge','ev_discharge','hp_power'):
        if name in central4:
            c=np.asarray(central4[name],float)
            v=np.vstack([x[name] for x in physical])
            if c.shape==v.shape:
                comparison.append(dict(control=name,RMS_kW=np.sqrt(np.mean((c-v)**2)),
                                       max_abs_kW=np.max(np.abs(c-v))))
    if comparison: display(pd.DataFrame(comparison).round(7))
    else: print('Legacy dispatch keys not directly aligned; costs reported above.')
except Exception as exc:
    central4=None
    print('Legacy centralized 4h solve unavailable:',repr(exc))
    print('This is NOT evidence of coincidence; fix legacy data compatibility before claiming it.')

Legacy centralized 4h time: 0.03 s


,Prosumer,Centralized 4h [EUR],Full feeder vGNE 4h [EUR],Screened vGNE 4h [EUR],Screened − centralized [EUR]
0,P1,11.050777,11.050777,11.050777,0.0
1,P2,54.985863,54.985863,54.985863,-0.0
2,P3,3.321077,3.321077,3.321077,0.0
3,P4,10.859313,10.859313,10.859313,-0.0


,control,RMS_kW,max_abs_kW
0,buy,1.000000e-07,3.000000e-07
1,sell,0.000000e+00,0.000000e+00
2,charge,0.000000e+00,0.000000e+00
3,discharge,0.000000e+00,0.000000e+00
4,ev_charge,0.000000e+00,0.000000e+00
5,ev_discharge,0.000000e+00,0.000000e+00
6,hp_power,0.000000e+00,0.000000e+00


## 4 — Identify provably redundant local rows by LP

We solve a per-prosumer, per-step linear program for each candidate row: maximize that row's left-hand side subject to **all other currently retained local rows**. A row is removed only if the other local inequalities imply it. Testing against only local rows is conservative: it may miss reductions implied by dynamics or state bounds. It is nevertheless safe when the LP certifies redundancy. Sequential removal prevents removing mutually redundant inequalities together. Optional; may require hundreds of small LPs.

In [9]:
def audit_and_prune_local(case, *, apply=False, tol=1e-7):
    g=case.game
    kw={name:getattr(g,name) for name in GAME_FIELDS}
    newA=[]; newb=[]; audit=[]
    for t in range(case.T):
        a_t=[]; b_t=[]
        for i in range(case.N):
            A=np.asarray(g.C_loc[t][i],float).copy()
            b=np.asarray(g.b_loc[t][i],float).copy()
            initial=len(b)
            keep=list(range(initial))
            failures=0
            for j in range(initial):
                if j not in keep: continue
                others=[k for k in keep if k!=j]
                if not others: continue
                # No variable bounds here; all physical bounds are in A.
                lp=linprog(-A[j],A_ub=A[others],b_ub=b[others],
                           bounds=[(None,None)]*A.shape[1],method='highs')
                if lp.status==0 and (-lp.fun <= b[j]+tol):
                    keep.remove(j)
                elif lp.status not in (0,3):
                    failures+=1  # Infeasible/numerical/other: do not drop
            audit.append(dict(t=t,prosumer=f'P{i+1}',
                  initial=initial,remaining=len(keep),removed=initial-len(keep),
                  LP_unresolved=failures))
            a_t.append(A[keep].copy()); b_t.append(b[keep].copy())
        newA.append(a_t);newb.append(b_t)
    report=pd.DataFrame(audit)
    if not apply: return case,report
    kw.update(C_loc=newA,b_loc=newb)
    return replace(case,game=DynLQGameTV(**kw)),report

start=time.perf_counter()
_,local_report4=audit_and_prune_local(reduced4,apply=False)
print('Local LP audit time:',round(time.perf_counter()-start,2),'s')
display(local_report4.groupby('prosumer')[['initial','remaining','removed','LP_unresolved']].sum())
print('Local rows:',local_report4.initial.sum(),'→',local_report4.remaining.sum())

Local LP audit time: 1.68 s


,initial,remaining,removed,LP_unresolved
prosumer,,,,
P1,192,144,48,0
P2,192,148,44,0
P3,192,132,60,0
P4,192,144,48,0


Local rows: 768 → 568


## 5 — Apply the certified local reduction, then validate the vGNE

This only removes per-step local inequalities already implied by the other local inequalities; it does **not** reduce β, because β depends on shared rows. It can still reduce the size of the static subproblems. Because tolerance is numerical, inspect maximum violation and cost differences.

In [10]:
local4, applied_audit4=audit_and_prune_local(reduced4,apply=True)
start=time.perf_counter()
sol_local4=solve_compact(local4,verbose=False)
print('LP-screened local vGNE time:',round(time.perf_counter()-start,2),'s')
J_local4=evaluated_costs(local4,sol_local4['u'])
print('Max cost Δ [EUR]:',float(np.max(np.abs(J_local4-J_reduced4))))
print('Control RMS Δ [kW]:',float(np.sqrt(np.mean((sol_local4['u']-sol_reduced4['u'])**2))))
print('Violation against PRE-REDUCTION model:',
      model_max_violation(reduced4,sol_local4['u']))
print('Local rows remaining:',sum(local4.game.C_loc[t][i].shape[0]
                                 for t in range(local4.T) for i in range(local4.N)))

LP-screened local vGNE time: 0.02 s
Max cost Δ [EUR]: 3.300439921360976e-10
Control RMS Δ [kW]: 7.457556546190256e-11
Violation against PRE-REDUCTION model: 1.7763568394002505e-15
Local rows remaining: 568


## 6 — Trade sensitivity at 4h: all connected three-edge trees

Dropping an edge removes two directed trade controls and the pair of reciprocity inequalities per period; this is **not exact in general**. Each tree is re-solved with its own feeder screening. Its vGNE costs and network feasibility are compared against the **same 4h four-edge cycle**. If one tree matches well, it is a candidate for explicit GNE enumeration, not proof of unchanged GNE-set fairness.

In [11]:
def connected(n,edges):
    seen={0}
    for _ in range(n):
        seen |= {a for a,b in edges if b in seen}
        seen |= {b for a,b in edges if a in seen}
    return len(seen)==n

trees=[tuple(combo) for combo in itertools.combinations(edges4,len(edges4)-1)
       if connected(len(data4['demand']),combo)]
print('Trees:',trees)
trials=[]
trade_cases={}
for k,edges in enumerate(trees):
    data_tree=copy.deepcopy(data4);data_tree['edges']=list(edges)
    full_tree=build_game(data_tree,grid=grid,clearing_epsilon_kw=0.)
    certification=certify_feeder_constraints(full_tree)
    allowed=certification.loc[~certification['Certified redundant'],'Label'].tolist()
    tree,_=rebuild_constraints(full_tree,keep_labels=allowed,clean_locals=True,
                               localize_single_agent=True)
    try:
        tic=time.perf_counter();sol=solve_compact(tree,verbose=False)
        J=evaluated_costs(tree,sol['u'])
        trials.append(dict(tree=k,edges=str(edges),variables=tree.T*sum(tree.game.nu),
           shared=tree.T*tree.game.m_u,beta=(tree.N-1)*tree.T*tree.game.m_u,
           max_cost_delta_EUR=float(np.max(np.abs(J-J_full4))),
           total_cost_delta_EUR=float(J.sum()-J_full4.sum()),
           max_violation=float(model_max_violation(full_tree,sol['u'])),
           runtime_s=round(time.perf_counter()-tic,2)))
        trade_cases[k]=(tree,sol,edges)
    except Exception as exc:
        print(f'Tree {k} failed:',repr(exc))
display(pd.DataFrame(trials).sort_values('max_cost_delta_EUR') if trials else 'No successful tree results')

Trees: [((0, 1), (1, 2), (2, 3)), ((0, 1), (1, 2), (0, 3)), ((0, 1), (2, 3), (0, 3)), ((1, 2), (2, 3), (0, 3))]


,tree,edges,variables,shared,beta,max_cost_delta_EUR,total_cost_delta_EUR,max_violation,runtime_s
3,3,"((1, 2), (2, 3), (0, 3))",240,36,108,4.380403e-10,6.770051e-11,2.819966e-14,0.01
1,1,"((0, 1), (1, 2), (0, 3))",240,36,108,6.289977e-10,5.206431e-10,4.551914e-14,0.01
0,0,"((0, 1), (1, 2), (2, 3))",240,36,108,1.552600e-09,6.632064e-10,3.613776e-14,0.02
2,2,"((0, 1), (2, 3), (0, 3))",240,36,108,2.181096e-09,5.192391e-09,3.241851e-14,0.01


# Inspect Julia

In [13]:
from pathlib import Path
from juliacall import Main as jl
from p2p.setup import to_static_gnep

# Inspect the Julia package actually loaded
jl.seval("import ParametricDAQP")

src = Path(str(
    jl.seval("string(pathof(ParametricDAQP))")
))

print("Julia package:", src)
print("Source directory:", src.parent)

for filename, patterns in {
    "mpsolve.jl": ["compute_AS0(", "get_ignore_masks(", "print_ws("],
    "utils.jl": ["function compute_AS0", "function get_ignore_masks"],
}.items():
    path = src.parent / filename
    print(f"\n--- {filename} ---")

    if not path.exists():
        print("Not found in this package version")
        continue

    lines = path.read_text().splitlines()

    for i, line in enumerate(lines, 1):
        if any(p in line for p in patterns):
            print(f"Line {i}: {line.strip()}")

# Condensed dimensions, including state constraints
case = (
    gne4 if "gne4" in globals()
    else local4 if "local4" in globals()
    else reduced4
)

_, _, gp = to_static_gnep(case)

n_local = sum(A.shape[0] for A in gp.A_loc)

print("\n=== CONDENSED GAME ===")
print("Decision variables:", sum(gp.n))
print("Local inequalities:", n_local)
print("Shared inequalities:", gp.m_sh)
print("Total condensed inequalities:", n_local + gp.m_sh)
print("Beta dimension:", (gp.N - 1) * gp.m_sh)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Julia package: /Users/catharinahente/.julia/packages/ParametricDAQP/WwDLQ/src/ParametricDAQP.jl
Source directory: /Users/catharinahente/.julia/packages/ParametricDAQP/WwDLQ/src

--- mpsolve.jl ---
Line 54: AS0 = compute_AS0(prob,Θ)
Line 95: ignore_masks = get_ignore_masks(prob,Θ,typeof(as0),id_cands,ws)
Line 117: opts.verbose>0 && print_ws(ws,(j+=1))

--- utils.jl ---
Line 357: function compute_AS0(mpLDP::MPLDP,Θ)
Line 378: function compute_AS0(mpp::Union{MPQP,MPVI},Θ)
Line 448: function get_ignore_masks(mpp,Θ,UIntX,id_cands,ws)

=== CONDENSED GAME ===
Decision variables: 264
Local inequalities: 658
Shared inequalities: 48
Total condensed inequalities: 706
Beta dimension: 144


In [ ]:
import os, sys, json, time, pickle, signal, subprocess, textwrap, importlib
from datetime import datetime
from pathlib import Path
from dataclasses import replace
import numpy as np
import psutil
from dynect import DynLQGameTV

# ============================================================
# SETTINGS
# ============================================================

EPS = 0.01
BETA = 5.0
REGION_LIMIT = 100
TIME_LIMIT_MIN = 15
MAX_RAM_GB = 5.6

# ============================================================
# 1. USE OUR SMALLEST VALIDATED 4-HOUR MODEL
# ============================================================

base4 = local4 if "local4" in globals() else reduced4

assert base4.N == 4 and base4.T == 6
assert all(s.startswith("edge_") for s in base4.shared_labels)
assert all(np.allclose(b, 0) for b in base4.game.b_u)

# Preserve all existing dynamics, costs and local constraints.
# Relax ONLY the bilateral reciprocity inequalities.

fields = (
    "A", "B", "Q", "R", "P", "c", "q", "r", "p",
    "C_x", "b_x", "C_loc", "b_loc", "C_u", "b_u"
)

kw = {name: getattr(base4.game, name) for name in fields}

kw["b_u"] = [
    np.full_like(np.asarray(b, dtype=float), EPS)
    for b in base4.game.b_u
]

gne4 = replace(base4, game=DynLQGameTV(**kw))

shared = sum(len(b) for b in gne4.game.b_u)
n_beta = (gne4.N - 1) * shared

local_rows = sum(
    len(gne4.game.b_loc[t][i])
    for t in range(gne4.T)
    for i in range(gne4.N)
)

print("=== REDUCED GNE GAME ===")
print("Prosumers:           ", gne4.N)
print("Time steps:          ", gne4.T)
print("Decision variables:  ", gne4.T * sum(gne4.game.nu))
print("Shared inequalities: ", shared)
print("Local inequalities:  ", local_rows)
print("Beta dimension:      ", n_beta)
print("Trade tolerance:     ", EPS, "kW")

assert shared == 48 and n_beta == 144

# Check that relaxing reciprocity does not substantially
# change the previously validated vGNE.

if "solve_compact" in globals() and "evaluated_costs" in globals():
    if "local4" in globals() and "sol_local4" in globals():
        sol_reference = sol_local4
    elif "sol_reduced4" in globals():
        sol_reference = sol_reduced4
    else:
        sol_reference = solve_compact(base4, verbose=False)

    sol_eps = solve_compact(gne4, verbose=False)

    print("\n=== EFFECT OF EPSILON RELAXATION ===")
    print("Max cost difference [EUR]:",
          np.max(np.abs(
              evaluated_costs(gne4, sol_eps["u"])
              - evaluated_costs(base4, sol_reference["u"])
          )))
    print("Control RMS difference [kW]:",
          np.sqrt(np.mean(
              (sol_eps["u"] - sol_reference["u"]) ** 2
          )))

# ============================================================
# 2. CREATE A SEPARATE EXPERIMENT
# ============================================================

p2p_root = Path(
    importlib.import_module("p2p.setup").__file__
).resolve().parents[1]

dynect_root = Path(
    importlib.import_module("dynect").__file__
).resolve().parents[1]

run_dir = (
    p2p_root / ".gne_experiments" /
    datetime.now().strftime("trial4h_%Y%m%d_%H%M%S_%f")
)
run_dir.mkdir(parents=True)

input_path = run_dir / "game.pkl"
worker_path = run_dir / "worker.py"
log_path = run_dir / "solver.log"
summary_path = run_dir / "summary.json"
output_path = run_dir / "gne_set.pkl"

with input_path.open("wb") as f:
    pickle.dump(gne4, f)

# ============================================================
# 3. WORKER: JULIA + REGION ENUMERATION
# ============================================================

worker_code = r"""
import json, pickle, sys, time, importlib
from pathlib import Path
import numpy as np
from p2p.setup import to_static_gnep

gm = importlib.import_module("dynect.solvers.gne")

input_path, summary_path, output_path = map(Path, sys.argv[1:4])
beta = float(sys.argv[4])
region_limit = int(sys.argv[5])

def log(*args):
    print(time.strftime("%H:%M:%S"), *args, flush=True)

# Report entry and exit of Julia's multiparametric solver.
original_mpsolve = gm._julia.mpsolve

def traced_mpsolve(*args, **kwargs):
    log("ENTER Julia ParametricDAQP.mpsolve")
    t0 = time.monotonic()
    result = original_mpsolve(*args, **kwargs)
    log("EXIT Julia mpsolve after",
        round(time.monotonic() - t0, 1), "seconds")
    return result

gm._julia.mpsolve = traced_mpsolve

with input_path.open("rb") as f:
    case = pickle.load(f)

log("Constructing StaticGNEP...")

mp, avi, gnep = to_static_gnep(case)

n_beta = (gnep.N - 1) * gnep.m_sh

log(
    "StaticGNEP ready:",
    "decisions =", sum(gnep.n),
    "shared =", gnep.m_sh,
    "beta =", n_beta
)

assert n_beta == 144

log(
    "Starting GNESetSolver:",
    "verbose=2",
    "region_limit =", region_limit,
    "beta =", beta
)

solver = gm.GNESetSolver(
    gnep,
    beta_lb=np.full(n_beta, -beta),
    beta_ub=np.full(n_beta, beta),
    region_limit=region_limit,
    verbose=2,
)

summary = {
    "status": str(solver.status),
    "gne_regions": len(solver.gne_set.regions),
    "beta_dimension": n_beta,
}

summary_path.write_text(json.dumps(summary, indent=2))

log("SOLVER RETURNED:", summary)

with output_path.open("wb") as f:
    pickle.dump(solver.gne_set, f)

log("GNE SET SAVED:", output_path)
"""

worker_path.write_text(textwrap.dedent(worker_code))

# ============================================================
# 4. PACKAGE PATHS
# ============================================================

env = os.environ.copy()

env["PYTHONPATH"] = os.pathsep.join([
    str(p2p_root),
    str(dynect_root),
    env.get("PYTHONPATH", ""),
])
env["PYTHONUNBUFFERED"] = "1"

# ============================================================
# 5. MEMORY-PROTECTED EXECUTION
# ============================================================

print("\n=== STARTING ENUMERATION ===")
print("Region limit:", REGION_LIMIT)
print("Beta bounds: ±", BETA)
print("Time limit:", TIME_LIMIT_MIN, "minutes")
print("RAM watchdog:", MAX_RAM_GB, "GB")
print("Experiment folder:", run_dir, flush=True)

started = time.monotonic()
last_report = started
peak_gb = 0.0
stop_reason = None

with log_path.open("w") as logfile:

    proc = subprocess.Popen(
        [
            sys.executable, "-u",
            str(worker_path),
            str(input_path),
            str(summary_path),
            str(output_path),
            str(BETA),
            str(REGION_LIMIT),
        ],
        cwd=p2p_root,
        env=env,
        stdout=logfile,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    offset = 0

    while proc.poll() is None:
        time.sleep(1)

        memory_bytes = 0
        cpu_seconds = 0

        try:
            parent = psutil.Process(proc.pid)
            processes = [parent] + parent.children(recursive=True)

            for p in processes:
                try:
                    memory_bytes += p.memory_info().rss
                    cpu = p.cpu_times()
                    cpu_seconds += cpu.user + cpu.system
                except (psutil.NoSuchProcess, psutil.AccessDenied):
                    pass

        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

        peak_gb = max(peak_gb, memory_bytes / 1024**3)
        elapsed = time.monotonic() - started

        if peak_gb > MAX_RAM_GB:
            stop_reason = "RAM watchdog"

        elif elapsed > TIME_LIMIT_MIN * 60:
            stop_reason = "Time watchdog"

        if stop_reason:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            break

        if time.monotonic() - last_report >= 30:

            print(
                f"[{elapsed/60:.1f} min] "
                f"RAM peak={peak_gb:.2f} GB; "
                f"CPU time={cpu_seconds:.0f} s",
                flush=True,
            )

            with log_path.open(errors="replace") as f:
                f.seek(offset)
                new_output = f.read()
                offset = f.tell()

            if new_output:
                print(new_output[-3000:], flush=True)

            last_report = time.monotonic()

    proc.wait()

# ============================================================
# 6. RESULTS
# ============================================================

elapsed = time.monotonic() - started

print("\n=== EXECUTION SUMMARY ===")
print("Exit code:", proc.returncode)
print("Elapsed [min]:", round(elapsed / 60, 2))
print("Peak RAM [GB]:", round(peak_gb, 2))
print("Stop reason:", stop_reason)

if summary_path.exists():
    print("\n", summary_path.read_text())
    print("Saved GNE set:", output_path)
else:
    print("\nNo completed GNE set.")
    print("Solver log:")
    print(log_path.read_text(errors="replace")[-4000:])

print("\nExperiment directory:", run_dir)

SyntaxError: invalid syntax (2117213517.py, line 182)